## Matrix-Matrix Multiplication

In [3]:
import torch, time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N = 4096
A = torch.rand(N, N, device=device)
B = torch.rand(N, N, device=device)
torch.cuda.synchronize()
t0 = time.time()
C = torch.matmul(A, B)
torch.cuda.synchronize()
elapsed = time.time() - t0
flops = 2*N**3/elapsed
print(f"Size={N}, Time={elapsed:.4f}s, Throughput={flops/1e12:.2f} TFLOPs")


Size=4096, Time=0.0888s, Throughput=1.55 TFLOPs


## Vector Addition

In [1]:
import torch, time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N = 50_000_000
x = torch.rand(N, device=device)
y = torch.rand(N, device=device)
torch.cuda.synchronize()
t0 = time.time()
z = x + y
torch.cuda.synchronize()
elapsed = time.time() - t0
bytes = 2 * x.nelement() * x.element_size()  # bytes read+written
bandwidth = bytes/elapsed/1e9  # GB/s
print(f"N={N}, Time={elapsed:.4f}s, Bandwidth={bandwidth:.2f} GB/s")


N=50000000, Time=0.0226s, Bandwidth=17.66 GB/s


## Scalar-Vector Multiplication

In [2]:
import torch, time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N = 100_000_000
x = torch.rand(N, device=device)
scalar = 3.14
torch.cuda.synchronize()
t0 = time.time()
y = x * scalar
torch.cuda.synchronize()
elapsed = time.time() - t0
bytes = 2 * x.nelement() * x.element_size()
bandwidth = bytes/elapsed/1e9
print(f"N={N}, Time={elapsed:.4f}s, Bandwidth={bandwidth:.2f} GB/s")


N=100000000, Time=0.0146s, Bandwidth=54.61 GB/s


## Elementwise Matrix Addition

In [3]:
import torch, time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N = 8192
A = torch.rand(N, N, device=device)
B = torch.rand(N, N, device=device)
torch.cuda.synchronize()
t0 = time.time()
C = A + B
torch.cuda.synchronize()
elapsed = time.time() - t0
bytes = 2 * A.nelement() * A.element_size()
bandwidth = bytes/elapsed/1e9
print(f"N={N}, Time={elapsed:.4f}s, Bandwidth={bandwidth:.2f} GB/s")


N=8192, Time=0.0036s, Bandwidth=148.10 GB/s


## GPU Memory Copy (Device-to-Device)

In [4]:
import torch, time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N = 128*1024*1024  # 128M elements
a = torch.rand(N, device=device)
torch.cuda.synchronize()
t0 = time.time()
b = a.clone()   # copy on GPU
torch.cuda.synchronize()
elapsed = time.time() - t0
bytes = a.nelement() * a.element_size()
bandwidth = bytes/elapsed/1e9
print(f"Size={bytes/1e9:.2f}GB, Time={elapsed:.4f}s, Bandwidth={bandwidth:.2f} GB/s")


Size=0.54GB, Time=0.0054s, Bandwidth=100.16 GB/s


## Host→GPU Transfer (PCIe Upload)

In [5]:
import torch, time
import numpy as np
if torch.cuda.is_available():
    N = 200_000_000
    host = np.random.rand(N).astype(np.float32)
    t0 = time.time()
    device_tensor = torch.from_numpy(host).cuda()
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    bytes = host.nbytes
    bandwidth = bytes/elapsed/1e9
    print(f"Upload {bytes/1e9:.2f}GB, Time={elapsed:.4f}s, Bandwidth={bandwidth:.2f} GB/s")


Upload 0.80GB, Time=0.1773s, Bandwidth=4.51 GB/s


## GPU→Host Transfer (PCIe Download)

In [6]:
import torch, time
if torch.cuda.is_available():
    N = 200_000_000
    a = torch.rand(N, device='cuda')
    t0 = time.time()
    host_copy = a.cpu()
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    bytes = a.nelement() * a.element_size()
    bandwidth = bytes/elapsed/1e9
    print(f"Download {bytes/1e9:.2f}GB, Time={elapsed:.4f}s, Bandwidth={bandwidth:.2f} GB/s")


Download 0.80GB, Time=0.7848s, Bandwidth=1.02 GB/s


## Stream Triad (Scale-Add)

In [7]:
import torch, time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N = 50_000_000
A = torch.rand(N, device=device)
B = torch.rand(N, device=device)
alpha = 2.5
torch.cuda.synchronize()
t0 = time.time()
A = A + alpha * B
torch.cuda.synchronize()
elapsed = time.time() - t0
bytes = (2 * A.nelement() + A.nelement()) * A.element_size()  # 2 reads (A,B) + 1 write
bandwidth = bytes/elapsed/1e9
print(f"N={N}, Time={elapsed:.4f}s, Bandwidth={bandwidth:.2f} GB/s")


N=50000000, Time=0.0043s, Bandwidth=139.38 GB/s


## CPU Matrix Multiply (Compute-bound, CPU)

In [8]:
import numpy as np, time
N = 4000
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)
t0 = time.time()
C = A.dot(B)
elapsed = time.time() - t0
flops = 2*N**3/elapsed
print(f"(CPU) N={N}, Time={elapsed:.4f}s, Throughput={flops/1e12:.2f} TFLOPs")


(CPU) N=4000, Time=0.1868s, Throughput=0.69 TFLOPs


## CPU Memory Copy (Memory-bound, CPU)

In [9]:
import numpy as np, time
N = 200_000_000
x = np.random.rand(N).astype(np.float32)
t0 = time.time()
y = x.copy()
elapsed = time.time() - t0
bandwidth = x.nbytes/elapsed/1e9
print(f"(CPU) Copied {x.nbytes/1e9:.2f}GB in {elapsed:.4f}s, Bandwidth={bandwidth:.2f} GB/s")


(CPU) Copied 0.80GB in 0.5805s, Bandwidth=1.38 GB/s
